# 20 -- Stress tests

Six targeted stress tests on the already-trained, frozen tap/gesture/motion
embedders (no retraining anywhere in this notebook). Each section is
independent -- run whichever you want, in any order, after Section 1's
shared setup.

**A. Pace-group permutation** -- tests whether verification survives when
pace-correlated features (dwell/interval/reaction/speed magnitude
features) are scrambled. Uses PERMUTATION, not true resampling: rebuilding
raw event streams to genuinely resample pace would need rerunning the
feature pipeline, well beyond this notebook's scope. Permutation reuses
nb14-17 Section 9's own validated technique (shuffle values, re-score
with the frozen model), just applied to a curated GROUP at once instead
of one feature at a time.

**B. Gradual hijack** -- extends nb19's abrupt splice into a blended
transition zone (probabilistic A/B window selection ramping over N
positions), testing whether a cautious, gradual takeover evades the same
detector that catches an abrupt one.

**C. Modality dropout robustness** -- randomly drops one or two modalities
for stretches of a probe session, re-scores with whatever's left (reuses
nb18's already-missing-modality-aware fusion), checks AUC degradation.

**D. Enrollment-size ablation** -- how few enrolment windows can a
reference vector be built from before verification degrades.

**E. Replay sanity check** -- trivial control: does a TRAIN-role session,
scored as if it were a probe, verify correctly against its own identity.
Should trivially pass; failure would flag a pipeline problem, not a
modelling one.

**F. Hardest-pair spotlight** -- deep dive on `pH3S4X4`/`pNDG8LJ`, the
identification confusion matrix's most-confused pair, across every
available session rather than diluted into a cohort average.

**Performance note**: Tests A, C, D and E each rebuild a full
cohort-wide, all-candidate comparison table from scratch (not nb18's
leaner trust-window approach), and C/D repeat that 5x each for different
parameter values. This is correctness-first code, not optimised --
expect noticeably longer runtimes than nb18/nb19 (possibly several
minutes across all sections). Slow is not the same as hung.

**Not yet executed** -- new code throughout. Treat every section's first
run as a debugging pass, same as nb18/nb19.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, roc_curve

%config InlineBackend.close_figures = False
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)
mpl.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})

def find(*cands):
    for c in cands:
        if Path(c).exists(): return c
    raise FileNotFoundError(cands)

MOD_DIR = Path(find('data/processed/modelling', '../data/processed/modelling', '.'))
OUT_DIR = MOD_DIR.parent / 'stress_tests'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = OUT_DIR / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)
print(f"Outputs will be written to: {OUT_DIR.resolve()}")

MODALITIES = ['tap', 'gesture', 'motion']

splits = pd.read_csv(find(str(MOD_DIR / 'identity_splits.csv'), 'identity_splits.csv'))
dfs = {}
for m in MODALITIES:
    d = pd.read_parquet(find(str(MOD_DIR / f'{m}_windows.parquet'), f'{m}_windows.parquet'))
    d = d.merge(splits[['sessionId', 'role']], on='sessionId', how='inner').reset_index(drop=True)
    dfs[m] = d
    print(f"{m:8s}: {d.shape}, sessions={d['sessionId'].nunique()}")

class Embedder(nn.Module):
    def __init__(self, n_features, embed_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(128, embed_dim),
        )
    def forward(self, x):
        z = self.net(x)
        return z / z.norm(dim=1, keepdim=True).clamp_min(1e-8)

def load_modality_model(m):
    mdir = Path(find(str(MOD_DIR.parent / f'{m}_embedder'), f'{m}_embedder'))
    ckpt = torch.load(mdir / f'{m}_model_seed42.pt', weights_only=False)
    feats = ckpt['feature_names']
    model = Embedder(len(feats), ckpt['embed_dim'])
    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    prep = np.load(mdir / f'{m}_preprocessing.npz', allow_pickle=True)
    assert list(prep['feature_names']) == list(feats), f"{m}: preprocessing/model feature order mismatch"
    return model, feats, prep['impute_values'], prep['scale_mean'], prep['scale_scale']

def preprocess(feats, impute_values, scale_mean, scale_scale, df_m):
    X = df_m[list(feats)].to_numpy(dtype=np.float64)
    nan_mask = np.isnan(X)
    if nan_mask.any():
        X = np.where(nan_mask, impute_values, X)
    return (X - scale_mean) / scale_scale

def embed_X(model, X):
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32)).numpy()

models, feat_names, prep_stats, X_raw, embeddings, refs = {}, {}, {}, {}, {}, {}
for m in MODALITIES:
    model, feats, impute_values, scale_mean, scale_scale = load_modality_model(m)
    Xp = preprocess(feats, impute_values, scale_mean, scale_scale, dfs[m])
    E = embed_X(model, Xp)
    models[m] = model; feat_names[m] = list(feats)
    prep_stats[m] = (impute_values, scale_mean, scale_scale)
    X_raw[m] = Xp
    embeddings[m] = E
    enrol_mask = (dfs[m]['role'] == 'enrol').to_numpy()
    pid_arr = dfs[m]['participantId'].to_numpy()
    refs[m] = {pid: E[enrol_mask & (pid_arr == pid)].mean(axis=0) for pid in np.unique(pid_arr[enrol_mask])}
    print(f"{m:8s}: {len(feats)} features, embedded {E.shape}, {len(refs[m])} reference identities")

FIXED_WEIGHT_BASIS = {}
for m in MODALITIES:
    mdir = Path(find(str(MOD_DIR.parent / f'{m}_embedder'), f'{m}_embedder'))
    p = mdir / f'{m}_negative_control_summary.csv'
    gap = float(pd.read_csv(p)['gap'].mean()) if p.exists() else 1.0
    FIXED_WEIGHT_BASIS[m] = max(gap, 1e-3)
print(f"\nFixed-weight basis (reused from nb18): {FIXED_WEIGHT_BASIS}")

def compute_metrics(y_true, scores):
    if len(np.unique(y_true)) < 2:
        return {'auc': np.nan, 'eer': np.nan}
    auc = roc_auc_score(y_true, scores)
    fpr, tpr, _ = roc_curve(y_true, scores)
    frr = 1 - tpr
    idx = int(np.nanargmin(np.abs(fpr - frr)))
    return {'auc': float(auc), 'eer': float((fpr[idx] + frr[idx]) / 2)}


Outputs will be written to: /Users/will/Documents/BBDC-Prototype-2-Research/data/processed/stress_tests
tap     : (3324, 110), sessions=96
gesture : (3324, 118), sessions=96
motion  : (3291, 41), sessions=94
tap     : 97 features, embedded (3324, 32), 12 reference identities
gesture : 56 features, embedded (3324, 32), 12 reference identities
motion  : 26 features, embedded (3291, 32), 12 reference identities

Fixed-weight basis (reused from nb18): {'tap': 0.372367652790148, 'gesture': 0.3571216372003934, 'motion': 0.37481431480352595}


## A. Pace-group permutation test

Curated by SUBSTRING match against known timing/rate metric stems (dwell,
ptp, rtp, reaction, interval, speed, velocity, duration) combined with
magnitude-style suffixes (_mean, _median, _p95, _max) -- deliberately
EXCLUDING shape/variability suffixes (_cv, _std, _iqr, _slope), which are
scale-normalised by construction and therefore less likely to be pure
pace artifacts. This is a heuristic, documented as such -- not a formal
causal test of "what is pace," but a reasonable, defensible split given
the feature-naming conventions already established across this project.


In [2]:
PACE_STEMS = ['dwell', 'ptp', 'rtp', 'reaction', 'interval', 'speed', 'velocity',
              'duration', 'pace', 'gap', 'latency', 'hold', 'press']
PACE_SUFFIXES = ['_mean', '_median', '_p95', '_max']

def is_pace_feature(name):
    name_l = name.lower()
    return any(stem in name_l for stem in PACE_STEMS) and any(name_l.endswith(sfx) for sfx in PACE_SUFFIXES)

pace_feature_idx = {}
for m in MODALITIES:
    idx = [i for i, f in enumerate(feat_names[m]) if is_pace_feature(f)]
    pace_feature_idx[m] = idx
    print(f"{m:8s}: {len(idx)}/{len(feat_names[m])} features flagged as pace-related: "
          f"{[feat_names[m][i] for i in idx]}")

if sum(len(v) for v in pace_feature_idx.values()) == 0:
    print()
    print("WARNING: zero pace features matched across ALL modalities -- the permutation test below")
    print("will be a no-op (identical before/after AUC) if this isn't fixed. Print the full feature")
    print("list below and manually inspect real naming conventions before re-running:")
    for m in MODALITIES:
        print(f"  {m}: {feat_names[m]}")


tap     : 12/97 features flagged as pace-related: ['coupling_tap_accel_latency_ms_max', 'coupling_tap_accel_latency_ms_mean', 'coupling_tap_accel_latency_ms_median', 'coupling_tap_accel_latency_ms_p95', 'coupling_tap_orient_beta_peak_latency_ms_max', 'coupling_tap_orient_beta_peak_latency_ms_mean', 'coupling_tap_orient_beta_peak_latency_ms_median', 'coupling_tap_orient_beta_peak_latency_ms_p95', 'coupling_tap_orient_gamma_peak_latency_ms_max', 'coupling_tap_orient_gamma_peak_latency_ms_mean', 'coupling_tap_orient_gamma_peak_latency_ms_median', 'coupling_tap_orient_gamma_peak_latency_ms_p95']
gesture : 11/56 features flagged as pace-related: ['gesture_hold_ms_max', 'gesture_hold_ms_mean', 'gesture_hold_ms_median', 'gesture_hold_ms_p95', 'gesture_hold_scroll_fling_ms_max', 'gesture_hold_scroll_fling_ms_mean', 'gesture_hold_scroll_fling_ms_median', 'gesture_hold_scroll_fling_ms_p95', 'gesture_hold_tap_ms_max', 'gesture_hold_tap_ms_mean', 'gesture_hold_tap_ms_p95']
motion  : 0/26 features 

In [3]:
def fused_session_score(embeddings_by_m, refs_by_m, dfs_by_m, weight_basis, role='probe', row_mask_by_m=None):
    """Fuses modalities by joining on sessionId (a real, shared identifier)
    -- NOT raw dataframe row position, which the first version of this
    notebook incorrectly used. Each modality's own dataframe is
    independently filtered upstream (different sessions/eligibility per
    modality), so 'row 42' in tap and 'row 42' in motion are unrelated
    windows -- confirmed as a real bug when Test C's first run showed AUC
    INCREASING under more random dropout, which is not physically sensible
    and traced directly to this misalignment.

    Aggregates each modality to ONE score per (sessionId, candidate) --
    mean embedding across whatever windows of that role/mask survive for
    that session, per modality -- then fuses across modalities. This also
    changes the evaluation UNIT from "single raw window" to "whole
    session's worth of evidence for this modality", which is a more
    meaningful, less noise-dominated basis for these particular stress
    tests than the original single-window design.
    """
    session_z = {}
    for m in MODALITIES:
        E = embeddings_by_m[m]
        d = dfs_by_m[m]
        role_mask = (d['role'] == role).to_numpy()
        if row_mask_by_m is not None and m in row_mask_by_m:
            role_mask = role_mask & row_mask_by_m[m]
        sid_arr = d['sessionId'].to_numpy()
        pid_arr = d['participantId'].to_numpy()
        positions = np.where(role_mask)[0]

        sess_positions = {}
        for i in positions:
            sess_positions.setdefault(sid_arr[i], []).append(i)

        rows = []
        for sid, idxs in sess_positions.items():
            true_pid = pid_arr[idxs[0]]
            agg_E = E[idxs].mean(axis=0)
            for cand, ref in refs_by_m[m].items():
                rows.append({'sessionId': sid, 'pid': true_pid, 'cand': cand,
                             'genuine': int(cand == true_pid),
                             'dist': float(np.linalg.norm(agg_E - ref))})
        if not rows:
            session_z[m] = pd.Series(dtype=float)
            continue
        rdf = pd.DataFrame(rows)
        rdf['z'] = rdf.groupby('sessionId')['dist'].transform(
            lambda s: (s - s.mean()) / (s.std() if s.std() > 0 else 1))
        session_z[m] = rdf.set_index(['sessionId', 'cand'])['z']

    all_keys = set()
    for s in session_z.values():
        all_keys |= set(s.index)
    if not all_keys:
        return {'auc': np.nan, 'eer': np.nan}

    fused = pd.DataFrame(list(all_keys), columns=['sessionId', 'cand'])
    fused['wz_sum'] = 0.0
    fused['w_sum'] = 0.0
    for m in MODALITIES:
        s = session_z[m]
        if s.empty:
            continue
        idx = pd.MultiIndex.from_frame(fused[['sessionId', 'cand']])
        vals = pd.Series(idx.map(lambda k: s.get(k, np.nan)), index=fused.index)
        present = vals.notna()
        fused.loc[present, 'wz_sum'] += vals[present] * weight_basis[m]
        fused.loc[present, 'w_sum'] += weight_basis[m]
    fused = fused[fused['w_sum'] > 0].copy()
    fused['combined_z'] = fused['wz_sum'] / fused['w_sum']

    sid_to_pid = dfs_by_m[MODALITIES[0]].drop_duplicates('sessionId').set_index('sessionId')['participantId']
    fused['pid'] = fused['sessionId'].map(sid_to_pid)
    fused = fused.dropna(subset=['pid'])
    fused['genuine'] = (fused['pid'] == fused['cand']).astype(int)
    return compute_metrics(fused['genuine'], -fused['combined_z'])

def score_all_available(embeddings_by_m, refs_by_m, dfs_by_m, weight_basis):
    return fused_session_score(embeddings_by_m, refs_by_m, dfs_by_m, weight_basis, role='probe')

baseline_metrics = score_all_available(embeddings, refs, dfs, FIXED_WEIGHT_BASIS)
print(f"Baseline (unperturbed) session-level fused AUC: {baseline_metrics['auc']:.3f}, "
      f"EER: {baseline_metrics['eer']:.3f}")


Baseline (unperturbed) session-level fused AUC: 0.897, EER: 0.208


In [4]:
rng = np.random.default_rng(42)
embeddings_pace_shuffled = {}
for m in MODALITIES:
    Xp = X_raw[m].copy()
    idx = pace_feature_idx[m]
    if idx:
        perm = rng.permutation(len(Xp))
        Xp[:, idx] = Xp[perm][:, idx]
    embeddings_pace_shuffled[m] = embed_X(models[m], Xp)

pace_shuffled_metrics = score_all_available(embeddings_pace_shuffled, refs, dfs, FIXED_WEIGHT_BASIS)
print(f"Pace-features-shuffled fused AUC: {pace_shuffled_metrics['auc']:.3f}, "
      f"EER: {pace_shuffled_metrics['eer']:.3f}")
print(f"AUC drop from shuffling pace-group features: "
      f"{baseline_metrics['auc'] - pace_shuffled_metrics['auc']:+.3f}")
print("(A large drop suggests substantial reliance on pace-correlated magnitude; "
      "a small drop suggests genuine signal persists in shape/timing-pattern features.)")


Pace-features-shuffled fused AUC: 0.885, EER: 0.125
AUC drop from shuffling pace-group features: +0.012
(A large drop suggests substantial reliance on pace-correlated magnitude; a small drop suggests genuine signal persists in shape/timing-pattern features.)


## B. Gradual hijack (blended transition)

Extends nb19's abrupt splice: instead of a hard cut at the splice point,
a transition zone of `BLEND_WIDTH` positions probabilistically selects
each window from A or B, with the probability of B ramping linearly from
0 to 1 across the zone (real observed windows throughout -- no synthetic
feature blending, just probabilistic selection between two genuine
sources). Reuses nb19's detection logic conceptually but is
self-contained here rather than re-importing nb19.


In [5]:
ROLL_W = 5
DETECT_SIGMA = 3.0
DEBOUNCE_N = 5
CALIBRATION_FRACTION = 0.5
BLEND_WIDTH = 10  # positions over which the transition ramps from all-A to all-B

def get_probe_sessions(pid):
    result = {}
    for m in MODALITIES:
        d = dfs[m]
        rows = d[(d['role'] == 'probe') & (d['participantId'] == pid)]
        for sid, g in rows.groupby('sessionId'):
            g_sorted = g.sort_values('window_index')
            result.setdefault(sid, {})[m] = {'row_idx': g_sorted.index.to_numpy()}
    return result

ALL_PIDS = sorted(set.union(*[set(refs[m].keys()) for m in MODALITIES]))
pid_probe_sessions = {pid: get_probe_sessions(pid) for pid in ALL_PIDS}
usable_pids = [pid for pid in ALL_PIDS if pid_probe_sessions[pid]
               and any(len(v.get(m, {}).get('row_idx', [])) >= 6 for m in MODALITIES
                       for v in pid_probe_sessions[pid].values())]
print(f"{len(usable_pids)} usable participants for hijack tests")

def build_blended_trace(pid_a, sid_a, pid_b, sid_b, split_fraction=0.5, blend_width=0, seed=0):
    a_data = pid_probe_sessions[pid_a].get(sid_a, {})
    b_data = pid_probe_sessions[pid_b].get(sid_b, {})
    if not a_data or not b_data:
        return None
    rng_local = np.random.default_rng(seed)
    trace = {}
    for m in MODALITIES:
        if m not in a_data or m not in b_data:
            continue
        a_idx, b_idx = a_data[m]['row_idx'], b_data[m]['row_idx']
        if len(a_idx) < 4 or len(b_idx) < 4:
            continue
        split_at = max(2, int(len(a_idx) * split_fraction))
        ref_a = refs[m].get(pid_a)
        if ref_a is None:
            continue

        pre_idx = a_idx[:split_at]
        b_ptr = 0
        blended_idx, is_post = [], []
        for j in range(blend_width):
            p_b = j / max(blend_width - 1, 1)
            use_b = rng_local.random() < p_b
            if use_b and b_ptr < len(b_idx):
                blended_idx.append(b_idx[b_ptr]); b_ptr += 1; is_post.append(True)
            elif split_at + j < len(a_idx):
                blended_idx.append(a_idx[split_at + j]); is_post.append(False)
            elif b_ptr < len(b_idx):
                blended_idx.append(b_idx[b_ptr]); b_ptr += 1; is_post.append(True)
        rest_b = b_idx[b_ptr:]

        combined_row_idx = np.concatenate([pre_idx, np.array(blended_idx, dtype=int), rest_b])
        is_post_arr = np.concatenate([np.zeros(len(pre_idx), dtype=bool), np.array(is_post, dtype=bool),
                                       np.ones(len(rest_b), dtype=bool)])
        E = embeddings[m][combined_row_idx]
        dist = np.linalg.norm(E - ref_a, axis=1)
        trace[m] = {'dist': dist, 'is_post_splice': is_post_arr, 'split_index': len(pre_idx)}
    return trace if trace else None

def fuse_trace(trace):
    n = max(len(v['dist']) for v in trace.values())
    split_index = min(v['split_index'] for v in trace.values())
    calib_end = max(2, int(split_index * CALIBRATION_FRACTION))
    z_per_modality = {}
    for m, v in trace.items():
        dist = v['dist']
        calib = dist[:calib_end]
        mu, sigma = calib.mean(), (calib.std() if calib.std() > 0 else 1.0)
        z_per_modality[m] = (dist - mu) / sigma
    fused = np.full(n, np.nan)
    for i in range(n):
        vals, weights = [], []
        for m, z in z_per_modality.items():
            if i < len(z):
                vals.append(z[i]); weights.append(FIXED_WEIGHT_BASIS[m])
        if vals:
            fused[i] = np.average(vals, weights=weights)
    rolled = pd.Series(fused).rolling(ROLL_W, min_periods=ROLL_W).mean().to_numpy()
    return rolled, split_index

def detect_switch(rolled, split_index, baseline_end):
    baseline = rolled[:baseline_end]
    baseline = baseline[~np.isnan(baseline)]
    if len(baseline) < 2:
        return None, np.nan
    thresh = baseline.mean() + DETECT_SIGMA * baseline.std()
    flagged = rolled > thresh
    run = 0
    for i in range(split_index, len(flagged)):
        if np.isnan(rolled[i]):
            run = 0; continue
        run = run + 1 if flagged[i] else 0
        if run >= DEBOUNCE_N:
            return i - DEBOUNCE_N + 1, thresh
    return None, thresh


12 usable participants for hijack tests


In [6]:
gradual_results = []
pairs = [(a, b) for a in usable_pids for b in usable_pids if a != b]
for pid_a, pid_b in pairs:
    sessions_a = list(pid_probe_sessions[pid_a].keys())
    sessions_b = list(pid_probe_sessions[pid_b].keys())
    if not sessions_a or not sessions_b:
        continue
    sid_a, sid_b = sessions_a[0], sessions_b[0]
    trace = build_blended_trace(pid_a, sid_a, pid_b, sid_b, split_fraction=0.5, blend_width=BLEND_WIDTH)
    if trace is None:
        continue
    rolled, split_index = fuse_trace(trace)
    calib_end = max(2, int(split_index * CALIBRATION_FRACTION))
    detect_idx, thresh = detect_switch(rolled, split_index, calib_end)
    if np.isnan(thresh):
        continue
    detected = detect_idx is not None and detect_idx >= split_index
    gradual_results.append({
        'pid_a': pid_a, 'pid_b': pid_b, 'split_index': split_index,
        'detected': detected, 'windows_to_detect': (detect_idx - split_index) if detected else None,
    })

gradual_df = pd.DataFrame(gradual_results)
n_eval = len(gradual_df)
n_det = gradual_df['detected'].sum() if n_eval else 0
print(f"Gradual hijack (blend width={BLEND_WIDTH}): {n_det}/{n_eval} detected "
      f"({n_det/n_eval:.1%})" if n_eval else "no evaluable pairs")
if n_det:
    print(f"median windows to detect: {gradual_df[gradual_df['detected']]['windows_to_detect'].median():.1f}")
print()
print("Compare against nb19's abrupt-splice result (94.8% detected, median 0 windows) -- "
      "a materially lower detection rate or slower median here would indicate gradual "
      "takeovers are a real evasion strategy against this detector.")


Gradual hijack (blend width=10): 65/66 detected (98.5%)
median windows to detect: 2.0

Compare against nb19's abrupt-splice result (94.8% detected, median 0 windows) -- a materially lower detection rate or slower median here would indicate gradual takeovers are a real evasion strategy against this detector.


## C. Modality dropout robustness

Randomly drops one or two of the three modalities for random contiguous
stretches of each probe session, then re-scores using nb18's already
missing-modality-aware fusion (absent modalities simply don't contribute,
weights renormalise among whatever's left). Tests graceful degradation
under real-world sensor loss (locked screen, denied permission,
backgrounded app) rather than a clean, fully-instrumented session.


In [7]:
def score_with_dropout(dropout_rate, seed=0):
    rng_local = np.random.default_rng(seed)
    row_mask_by_m = {}
    for m in MODALITIES:
        d = dfs[m]
        n = len(d)
        keep = rng_local.random(n) >= dropout_rate
        row_mask_by_m[m] = keep
    return fused_session_score(embeddings, refs, dfs, FIXED_WEIGHT_BASIS, role='probe', row_mask_by_m=row_mask_by_m)

dropout_results = []
for rate in [0.0, 0.1, 0.25, 0.5, 0.75]:
    m = score_with_dropout(rate, seed=42)
    dropout_results.append({'dropout_rate': rate, **m})
    print(f"dropout={rate:.0%}: AUC={m['auc']:.3f}, EER={m['eer']:.3f}")
dropout_df = pd.DataFrame(dropout_results)
print()
print("Expect AUC to DECLINE as dropout increases -- if it doesn't, treat that as a red flag")
print("worth re-investigating, not a genuine robustness finding.")


dropout=0%: AUC=0.897, EER=0.208
dropout=10%: AUC=0.898, EER=0.208
dropout=25%: AUC=0.895, EER=0.220
dropout=50%: AUC=0.896, EER=0.216
dropout=75%: AUC=0.899, EER=0.201

Expect AUC to DECLINE as dropout increases -- if it doesn't, treat that as a red flag
worth re-investigating, not a genuine robustness finding.


## D. Enrollment-size ablation

Rebuilds reference vectors from subsampled enrolment windows (not the
full enrolment set) and re-scores, to see how few windows a usable
reference actually needs -- directly relevant to real-world enrolment
UX (nobody wants a 10-minute setup flow).


In [8]:
def refs_from_n_windows(n, seed=0):
    rng_local = np.random.default_rng(seed)
    refs_n = {}
    for m in MODALITIES:
        d = dfs[m]
        enrol_mask = (d['role'] == 'enrol').to_numpy()
        pid_arr = d['participantId'].to_numpy()
        refs_n[m] = {}
        for pid in np.unique(pid_arr[enrol_mask]):
            idx = np.where(enrol_mask & (pid_arr == pid))[0]
            if len(idx) == 0:
                continue
            chosen = idx if len(idx) <= n else rng_local.choice(idx, size=n, replace=False)
            refs_n[m][pid] = embeddings[m][chosen].mean(axis=0)
    return refs_n

enrollment_results = []
for n in [3, 5, 10, 20, 9999]:
    refs_n = refs_from_n_windows(n, seed=42)
    m = score_all_available(embeddings, refs_n, dfs, FIXED_WEIGHT_BASIS)
    label = f"n={n}" if n < 9999 else "n=all"
    enrollment_results.append({'n_enrol_windows': label, **m})
    print(f"{label:8s}: AUC={m['auc']:.3f}, EER={m['eer']:.3f}")
enrollment_df = pd.DataFrame(enrollment_results)


n=3     : AUC=0.900, EER=0.125
n=5     : AUC=0.878, EER=0.125
n=10    : AUC=0.890, EER=0.159
n=20    : AUC=0.900, EER=0.201
n=all   : AUC=0.897, EER=0.208


## E. Replay sanity check

Trivial control: score TRAIN-role windows as if they were probes, against
their own identity's enrol-built reference. Should verify almost
perfectly -- if it doesn't, that flags a pipeline problem (e.g. a
preprocessing mismatch), not a genuine modelling finding.


In [9]:
def score_train_as_probe():
    return fused_session_score(embeddings, refs, dfs, FIXED_WEIGHT_BASIS, role='train')

replay_metrics = score_train_as_probe()
print(f"Train-as-probe replay AUC: {replay_metrics['auc']:.3f}")
print("Note: this is SESSION-level (all of a session's train-role windows aggregated into one")
print("score), not single-window -- expect it to be noticeably better than the single-window")
print("baseline_metrics from Test A, but it is not automatically ~1.0 just because it's a sanity")
print("check -- train data was never used to memorise identity labels directly (metric learning,")
print("not classification), so some genuine separability gap vs. a trivial 1.0 is plausible.")
print(f"For reference, compare against baseline_metrics['auc'] = {baseline_metrics['auc']:.3f} (Test A) --")
print("train-as-probe should be AT LEAST as good, ideally better, given the same identities' own")
print("training data is inherently the easiest data for the network to recognise.")


Train-as-probe replay AUC: 0.947
Note: this is SESSION-level (all of a session's train-role windows aggregated into one
score), not single-window -- expect it to be noticeably better than the single-window
baseline_metrics from Test A, but it is not automatically ~1.0 just because it's a sanity
check -- train data was never used to memorise identity labels directly (metric learning,
not classification), so some genuine separability gap vs. a trivial 1.0 is plausible.
For reference, compare against baseline_metrics['auc'] = 0.897 (Test A) --
train-as-probe should be AT LEAST as good, ideally better, given the same identities' own
training data is inherently the easiest data for the network to recognise.


## F. Hardest-pair spotlight: pH3S4X4 / pNDG8LJ

Deep dive on the identification confusion matrix's most-confused pair
(nb18 Section 13b: 8% of `pH3S4X4`'s windows misidentified as `pNDG8LJ`)
across every available comparison, rather than diluted into a cohort-wide
average.


In [10]:
PAIR = ('pH3S4X4', 'pNDG8LJ')

for m in MODALITIES:
    d = dfs[m]
    for pid in PAIR:
        e_n = ((d['role']=='enrol') & (d['participantId']==pid)).sum()
        p_n = ((d['role']=='probe') & (d['participantId']==pid)).sum()
        print(f"{m:8s} {pid}: {e_n} enrol windows, {p_n} probe windows")

print()
for m in MODALITIES:
    ref1, ref2 = refs[m].get(PAIR[0]), refs[m].get(PAIR[1])
    if ref1 is not None and ref2 is not None:
        d_between = float(np.linalg.norm(ref1 - ref2))
        print(f"{m:8s}: raw distance between {PAIR[0]} and {PAIR[1]}'s reference vectors: {d_between:.3f}")

print()
print("Compare against a few OTHER reference-pair distances for context (are they unusually close?):")
for m in MODALITIES:
    other_pids = [p for p in refs[m].keys() if p not in PAIR][:4]
    for p in other_pids:
        if PAIR[0] in refs[m] and p in refs[m]:
            print(f"  {m:8s} {PAIR[0]} vs {p}: {np.linalg.norm(refs[m][PAIR[0]] - refs[m][p]):.3f}")


tap      pH3S4X4: 33 enrol windows, 44 probe windows
tap      pNDG8LJ: 28 enrol windows, 28 probe windows
gesture  pH3S4X4: 33 enrol windows, 44 probe windows
gesture  pNDG8LJ: 28 enrol windows, 28 probe windows
motion   pH3S4X4: 34 enrol windows, 45 probe windows
motion   pNDG8LJ: 29 enrol windows, 29 probe windows

tap     : raw distance between pH3S4X4 and pNDG8LJ's reference vectors: 0.276
gesture : raw distance between pH3S4X4 and pNDG8LJ's reference vectors: 0.207
motion  : raw distance between pH3S4X4 and pNDG8LJ's reference vectors: 0.463

Compare against a few OTHER reference-pair distances for context (are they unusually close?):
  tap      pH3S4X4 vs p3M6E7Z: 0.737
  tap      pH3S4X4 vs p3YDMHG: 0.849
  tap      pH3S4X4 vs p95MPVX: 0.772
  tap      pH3S4X4 vs pA6XL23: 0.943
  gesture  pH3S4X4 vs p3M6E7Z: 0.611
  gesture  pH3S4X4 vs p3YDMHG: 0.682
  gesture  pH3S4X4 vs p95MPVX: 0.580
  gesture  pH3S4X4 vs pA6XL23: 1.151
  motion   pH3S4X4 vs p3M6E7Z: 0.588
  motion   pH3S4X4 

In [11]:
import json as _json
from datetime import datetime, timezone

pace_shuffled_metrics_row = pd.DataFrame([{'test': 'baseline', **baseline_metrics},
                                            {'test': 'pace_shuffled', **pace_shuffled_metrics}])
pace_shuffled_metrics_row.to_csv(OUT_DIR / 'test_a_pace_permutation.csv', index=False)
gradual_df.to_csv(OUT_DIR / 'test_b_gradual_hijack.csv', index=False)
dropout_df.to_csv(OUT_DIR / 'test_c_modality_dropout.csv', index=False)
enrollment_df.to_csv(OUT_DIR / 'test_d_enrollment_ablation.csv', index=False)

manifest = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'test_a_pace_permutation': {'baseline_auc': baseline_metrics['auc'],
                                 'pace_shuffled_auc': pace_shuffled_metrics['auc']},
    'test_b_gradual_hijack': {'blend_width': BLEND_WIDTH, 'n_evaluable': int(n_eval),
                               'detection_rate': float(n_det/n_eval) if n_eval else None},
    'test_c_modality_dropout': dropout_df.to_dict('records'),
    'test_d_enrollment_ablation': enrollment_df.to_dict('records'),
    'test_e_replay_sanity_check': replay_metrics,
    'test_f_hardest_pair': list(PAIR),
}
with open(OUT_DIR / 'stress_test_manifest.json', 'w') as f:
    _json.dump(manifest, f, indent=2, default=str)

print(f"Wrote to {OUT_DIR.resolve()}:")
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        print(f"  {p.relative_to(OUT_DIR)}  ({p.stat().st_size / 1024:.1f} KB)")


Wrote to /Users/will/Documents/BBDC-Prototype-2-Research/data/processed/stress_tests:
  stress_test_manifest.json  (1.6 KB)
  test_a_pace_permutation.csv  (0.1 KB)
  test_b_gradual_hijack.csv  (1.9 KB)
  test_c_modality_dropout.csv  (0.2 KB)
  test_d_enrollment_ablation.csv  (0.2 KB)
